# SAM 2.1 Promptable Segmentation Benchmark

This tutorial benchmarks SAM 2.1 promptable segmentation using point and box prompts.

**Models covered:** `sam2.1-hiera-tiny`, `sam2.1-hiera-small`  
**Backend:** PyTorch (CPU/CUDA)  
**License:** Apache-2.0

In [ ]:
# Cell 2: Setup
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'visionservex', '-q'])

import visionservex as vsx
import time
print(f'VisionServeX version: {vsx.__version__}')

from visionservex.sam2 import list_sam2_models, get_sam2_status
models = list_sam2_models()
print('Available SAM 2.1 models:')
for m in models:
    print(f'  {m["model_id"]:30s} status={m["status"]}')

In [ ]:
# Cell 3: Run sam2.1-hiera-tiny promptable segmentation
from visionservex.sam2 import run_sam2_promptable

tiny_result = run_sam2_promptable(
    model_id='sam2.1-hiera-tiny',
    prompt_type='point',
    point_coords=[[256, 256]],
    point_labels=[1],
    image_size=(512, 512)
)

print(f'Model        : sam2.1-hiera-tiny')
print(f'Status       : {tiny_result["status"]}')
print(f'Mask shape   : {tiny_result.get("mask_shape", "N/A")}')
print(f'IoU score    : {tiny_result.get("iou_score", "N/A")}')
print(f'Latency ms   : {tiny_result.get("latency_ms", "N/A")}')

In [ ]:
# Cell 4: Show mask result
import matplotlib.pyplot as plt
import numpy as np

if tiny_result.get('mask') is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(tiny_result['mask'].squeeze(), cmap='plasma')
    ax.set_title(f'SAM2.1-hiera-tiny mask  iou={tiny_result.get("iou_score", 0):.3f}')
    plt.tight_layout()
    plt.savefig('/tmp/sam21_tiny_mask.png', dpi=80)
    plt.show()
    print('Mask saved to /tmp/sam21_tiny_mask.png')
else:
    print(f'No mask returned — status: {tiny_result["status"]}')
    if 'message' in tiny_result:
        print(f'Message: {tiny_result["message"]}')

In [ ]:
# Cell 5: Compare sam2.1-hiera-tiny vs sam2.1-hiera-small
results = {}
for mid in ('sam2.1-hiera-tiny', 'sam2.1-hiera-small'):
    t0 = time.perf_counter()
    r = run_sam2_promptable(
        model_id=mid,
        prompt_type='point',
        point_coords=[[256, 256]],
        point_labels=[1],
        image_size=(512, 512)
    )
    elapsed = (time.perf_counter() - t0) * 1000
    results[mid] = {**r, 'wall_ms': elapsed}
    print(f'{mid}: status={r["status"]}  iou={r.get("iou_score","N/A")}  {elapsed:.0f} ms')

In [ ]:
# Cell 6: COCO subset approach for evaluation
from visionservex.sam2 import list_coco_subset_configs

configs = list_coco_subset_configs()
print('COCO subset evaluation configs:')
for c in configs:
    print(f'  subset={c["subset"]:20s} images={c["num_images"]}  annotations={c["num_annotations"]}')

print()
print('To run COCO subset mIoU evaluation:')
print('  from visionservex.sam2 import eval_sam2_coco_subset')
print('  metrics = eval_sam2_coco_subset("sam2.1-hiera-tiny", subset="val2017_100")')

## Next Steps

- See `03_sam21_video_tracking.ipynb` for SAM 2.1 video object tracking.
- For GroundingDINO+SAM pipeline, see `07_grounding_dino_sam_pipeline.ipynb`.
- Full COCO benchmark: `visionservex eval sam2.1-hiera-tiny --dataset coco-val2017`